In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Reading table from silver layer

In [0]:
df_gold = spark.read.table("albion_project.silver.prices")
display(df_gold.columns)

In [0]:
items_silver_df = spark.read.table("albion_project.silver.items")
display(items_silver_df.columns)


#Creating dimension table dim_item

In [0]:
dim_item_df = (
    df_gold
    .select(
        "item_id",
        "base_item_id",
        "enchant"
    )
    .dropDuplicates(["item_id"])
)
items_names_df = (
    items_silver_df
    .select(
        "item_id",
        "display_name_en",
        "display_name_pl"
    )
    .dropDuplicates(["item_id"])
)

dim_item_df = dim_item_df.join(
    items_names_df,
    "item_id",
    "left"
)
display(dim_item_df)
window = Window.orderBy("item_id")

dim_item_df = dim_item_df.withColumn(
    "item_key",
    F.row_number().over(window)
)

dim_item_df = dim_item_df.select(
    "item_key",
    "item_id",
    "base_item_id",
    "enchant",
    "display_name_en",
    "display_name_pl"
)

(
    dim_item_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("albion_project.gold.dim_item")
)


#Creating dimension table dim_city

In [0]:
dim_city_df = (
    df_gold
    .select(
        "city"
    )
    .distinct()
)

window = Window.orderBy("city")

dim_city_df = dim_city_df.withColumn(
    "city_key",
    F.row_number().over(window)
)

dim_city_df = dim_city_df.select(
    "city_key",
    "city"
)

(
dim_city_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("albion_project.gold.dim_city")
)

display(dim_city_df)

#Creating dimension table dim_quality

In [0]:
dim_quality_df = (
    df_gold
    .select(
        "quality"
    )
    .distinct()
)

window = Window.orderBy("quality")

dim_quality_df = dim_quality_df.withColumn(
    "quality_key",
    F.row_number().over(window)
)
dim_quality_df = dim_quality_df.withColumn(
    "quality_name",
    F.when(F.col("quality") == 1, "Normal")
     .when(F.col("quality") == 2, "Good")
     .when(F.col("quality") == 3, "Outstanding")
     .when(F.col("quality") == 4, "Excellent")
     .when(F.col("quality") == 5, "Masterpiece")
     .otherwise("Unknown")
)

dim_quality_df = dim_quality_df.select(
    "quality_key",
    "quality",
    "quality_name"
)

(
dim_quality_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("albion_project.gold.dim_quality")
)
display(dim_quality_df)

#Creating dimension table dim_date

In [0]:
date_range_df = (
    df_gold
    .agg(
        F.to_date(F.min("snapshot_at")).alias("start_date"),
        F.to_date(F.max("snapshot_at")).alias("end_date")
    ))


dim_date_df = (
    date_range_df
    .select(
        F.explode(
            F.sequence(
                F.col("start_date"),
                F.col("end_date"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
)

dim_date_df = (
    dim_date_df
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("full_date"))

)

dim_date_df = dim_date_df.select(
    "date_key",
    "full_date",
    "year",
    "quarter",
    "month",
    "month_name",
    "day",
    "day_of_week",
    "day_name",
    "week_of_year"
)

(
dim_date_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("albion_project.gold.dim_date")
)
display(dim_date_df)

#Creating fact table fact_market_price_history

In [0]:
display(df_gold.columns)

In [0]:
fact_market_price_history_df = (
    df_gold
    .withColumn(
        "snapshot_date",
        F.to_date("snapshot_at")
    )
    .join(dim_item_df, "item_id", "inner")
    .join(dim_city_df, "city", "inner")
    .join(dim_quality_df, "quality", "inner")
    .join(
        dim_date_df,
        F.col("snapshot_date") == F.col("full_date"),
        "inner"
    )
    .select(
        "item_key",
        "city_key",
        "quality_key",
        "date_key",
        "snapshot_at",
        "processed_at",
        "sell_price_min",
        "sell_price_min_date",
        "sell_price_max",
        "sell_price_max_date",
        "buy_price_min",
        "buy_price_min_date",
        "buy_price_max",
        "buy_price_max_date",
        "has_sell_offer",
        "has_buy_offer",
        "has_both_offers"
    )
)
display(fact_market_price_history_df)


#Testing gold df

In [0]:
fact_market_price_history_df.filter(
    F.col("item_key").isNull()
    | F.col("city_key").isNull()
    | F.col("quality_key").isNull()
    | F.col("date_key").isNull()
).count()

In [0]:
fact_market_price_history_df.groupBy(
    "item_key",
    "city_key",
    "quality_key",
    "snapshot_at"
).count().filter(
    F.col("count") > 1
).count()

#Writing gold df to gold table

In [0]:
(
    fact_market_price_history_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("albion_project.gold.fact_market_price_history")
)

In [0]:
dim_item_test_df = spark.table(
    "albion_project.gold.dim_item"
).select(
    "item_key",
    "item_id",
    "display_name_en"
)

fact_test_df = spark.table(
    "albion_project.gold.fact_market_price_history"
)

joined_test_df = fact_test_df.join(
    dim_item_test_df,
    on="item_key",
    how="left"
)

In [0]:
joined_test_df.filter(
    F.col("item_id").isNull()
).count()

In [0]:
dim_item_test_df.groupBy(
    "item_key"
).count().filter(
    F.col("count") > 1
).count()

In [0]:
fact_test_df.count()

In [0]:
joined_test_df.count()